[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ashakram05/ayeshaAkram-flyrank/blob/main/work/notebooks/w04_baseline_score.ipynb)


# ML-07 — Baseline Action Score

## Lane 2: Refresh / Content Opportunity Scoring

The goal of this baseline is to create a simple, transparent ranking of content items
that deserve human review first.

The rule is intentionally simple:

> Prioritize content that has meaningful search visibility and shows signs that it
> may benefit from attention.

This is a rule-based baseline, not a fitted machine-learning model.

The purpose is to create an honest benchmark that a later Week-5 model can attempt
to improve.

---

**Repair note (this version):** the rule itself is unchanged from the original Week-4
submission. What's new is Part 3 below — a properly labeled evaluation proxy and the
Precision@K / base-rate numbers the original notebook never computed, so Week 5 has an
actual number to beat, not just a rule to imitate.


In [ ]:
!pip -q install duckdb huggingface_hub

import duckdb
import pandas as pd
import numpy as np

from google.colab import userdata

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Get Hugging Face token from Colab Secrets
hf_token = userdata.get("flyrank")

if not hf_token:
    raise ValueError(
        "Hugging Face token not found. Add it to Colab Secrets as 'flyrank'."
    )

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET flyrank_hf (
    TYPE huggingface,
    TOKEN '{hf_token}'
)
""")

REL = "hf://datasets/FlyRank/internship-warehouse"

MARCH_PATH = (
    f"{REL}/fact_content_daily_performance/month=2026-03/*.parquet"
)

print("Connected to FlyRank warehouse.")


# 1. Signal Checks

Before encoding the baseline rule, I will check two signals that the rule will rely on.

The two signals are:

1. **Content freshness / staleness**
2. **Search visibility**

At least one signal should connect to a real FlyRank decision rule. Staleness is directly
connected to the refresh-oriented flags discussed in the session.

The purpose of these checks is not to prove causation. They are sanity checks asking
whether the signals show a useful directional relationship with the review opportunity.


In [ ]:
# Build the March decision snapshot.
#
# We use March 31 as the decision date and calculate recent signals
# only from information available on or before that date.

snapshot = con.sql(f"""
WITH daily AS (
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        gsc_data_available,
        ga4_data_available,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position,
        ga4_sessions,
        ga4_users
    FROM read_parquet('{MARCH_PATH}')
),

latest AS (
    SELECT *
    FROM daily
    WHERE report_date = DATE '2026-03-31'
)

SELECT *
FROM latest
""").df()

print("March 31 snapshot:", snapshot.shape)

display(snapshot.head())


### Signal 1 — Search visibility

I use GSC impressions as the visibility signal.

A content item with more search impressions has more observable search exposure.
This makes it more consequential to prioritize for review than a page with almost
no search visibility.

This is a directional signal, not proof that the page needs a refresh.


In [ ]:
# Bucket search impressions and inspect the distribution.

signal_1 = snapshot.copy()

signal_1["visibility_bucket"] = pd.cut(
    signal_1["gsc_impressions"],
    bins=[-1, 100, 500, 2000, np.inf],
    labels=["0-100", "101-500", "501-2000", "2000+"]
)

visibility_table = (
    signal_1
    .groupby("visibility_bucket", observed=False)
    .agg(
        n=("content_hash_id", "size"),
        mean_impressions=("gsc_impressions", "mean"),
        mean_clicks=("gsc_clicks", "mean")
    )
    .reset_index()
)

display(visibility_table)


### Verdict: CONFIRMED

The visibility buckets show that impressions represent materially different levels
of search exposure.

I therefore keep search visibility as one component of the baseline.

This does not mean high impressions automatically mean "refresh." It means that
reviewing a visible page has a clearer potential business impact than reviewing
a page with almost no observed search exposure.


### Signal 2 — Recent search activity

The warehouse does not contain a direct `last_updated` or `content_age` field.

Therefore, I will not manufacture a content-staleness feature.

Instead, I check recent search activity as a weaker observable signal.

This is related to the refresh decision, but it is not equivalent to content age.


In [ ]:
# Bucket recent impressions into low / medium / high visibility.

signal_2 = snapshot.copy()

signal_2["activity_bucket"] = pd.cut(
    signal_2["gsc_impressions"],
    bins=[-1, 0, 100, 1000, np.inf],
    labels=["0", "1-100", "101-1000", "1000+"]
)

activity_table = (
    signal_2
    .groupby("activity_bucket", observed=False)
    .agg(
        n=("content_hash_id", "size"),
        mean_clicks=("gsc_clicks", "mean"),
        mean_position=("gsc_avg_position", "mean")
    )
    .reset_index()
)

display(activity_table)


### Verdict: MIXED

Recent search activity is useful for separating content with meaningful observable
search exposure from content with little or no exposure.

However, it is not a direct measure of content freshness.

A page can have low search activity for many reasons unrelated to content age.

Therefore I will use it only as a supporting signal rather than claiming that it
proves a page is stale.


# 2. Baseline Rule

The baseline should remain simple enough for a non-technical reviewer to understand.

### Rule

Prioritize content that:

1. Has meaningful search visibility.
2. Has enough search activity to make review potentially useful.

The score combines the two conditions.

### Score

- High visibility + meaningful activity → higher priority
- High visibility + little activity → medium priority
- Very low visibility → lower priority

### Reason code

Each item receives exactly one primary reason code explaining why it received
its action.

### Action labels

- `review_first`
- `review_later`
- `low_priority`

This is deliberately not a fitted model. The score uses fixed human-readable
conditions so that Week 5 has a transparent baseline to beat.

**This rule is unchanged from the original Week-4 submission** — only the evaluation
below it (Parts 3–5) is new.


In [ ]:
baseline = snapshot.copy()

# Ensure numeric columns are numeric.
baseline["gsc_impressions"] = pd.to_numeric(
    baseline["gsc_impressions"], errors="coerce"
).fillna(0)

baseline["gsc_clicks"] = pd.to_numeric(
    baseline["gsc_clicks"], errors="coerce"
).fillna(0)

baseline["gsc_avg_position"] = pd.to_numeric(
    baseline["gsc_avg_position"], errors="coerce"
)

# Two transparent rule components.
visible = (baseline["gsc_impressions"] >= 500).astype(int)
active = (baseline["gsc_clicks"] >= 10).astype(int)

# Fixed, human-readable score.
baseline["score"] = (
    visible * 2 +
    active
)

# One reason code.
baseline["reason_code"] = np.select(
    [
        (visible == 1) & (active == 1),
        (visible == 1) & (active == 0)
    ],
    [
        "visible_and_active",
        "visible_but_low_clicks"
    ],
    default="low_visibility"
)

# Action label.
baseline["action"] = np.select(
    [
        baseline["score"] >= 3,
        baseline["score"] == 2
    ],
    [
        "review_first",
        "review_later"
    ],
    default="low_priority"
)

# Rank.
baseline = baseline.sort_values(
    ["score", "gsc_impressions"],
    ascending=[False, False]
).reset_index(drop=True)

baseline["rank"] = np.arange(1, len(baseline) + 1)

display(
    baseline[
        [
            "rank",
            "client_hash_id",
            "content_hash_id",
            "score",
            "reason_code",
            "action",
            "gsc_impressions",
            "gsc_clicks"
        ]
    ].head(20)
)


# 3. Future-Performance Evaluation Proxy

**Why a direct refresh label is unavailable:** the warehouse records what happened to
search visibility. It does not record whether anyone actually refreshed a page or
whether a refresh worked. Week 3's data contract already documented this: there is no
ground-truth `needs_refresh` field anywhere in the warehouse.

**Why a future-performance proxy is necessary:** to compute Precision@K at all, the
baseline rule needs *something* to be right or wrong about. Without a proxy, "the
baseline ranked these first" is just an assertion — there's no way to check whether the
ranking is doing anything useful.

**Exactly how the proxy is calculated:** for every content item in the March 31
decision snapshot, I look at its **April 2026** daily rows (the month immediately
following the decision date) and compute the average daily GSC impressions across
April. The proxy label is:

> `future_decline_proxy = 1` if April's average daily impressions is lower than the
> March 31 single-day value, else `0`.

**Why this future window is appropriate:** the warehouse's daily fact table is
partitioned by month (`month=YYYY-MM`), so April 2026 is the next complete partition
after the March decision snapshot — no gap, no overlap. Averaging across the whole
month (rather than comparing March 31 to April 1, a single day to single day) reduces
the single-day noise that would otherwise dominate a day-over-day comparison. April
2026 is fully contained in the warehouse's available date range (2025-01-27 through
2026-06-30), so this is real observed data, not a manufactured window.

**Why the proxy is NOT a ground-truth refresh label:** a drop in April's average
impressions can happen for many reasons that have nothing to do with content quality —
seasonality, a SERP layout change, a competitor outranking the page, or simply normal
volatility. The proxy answers "did this item's observed search visibility fall in the
following month," not "did this item need a refresh." Every claim below is phrased as
*the item experienced the future-performance condition used by the evaluation proxy*,
not as a refresh verdict.

**Limitation this introduces:** items with no April rows (e.g. a client's tracking
ended, or the content item stopped appearing) have no future outcome to check and are
dropped from the evaluated population — reported explicitly below, not silently
imputed. This is a survivorship limitation on the *evaluation*, not on the baseline
score itself (every March 31 row still gets a rank and an action).

In [ ]:
APRIL_PATH = f"{REL}/fact_content_daily_performance/month=2026-04/*.parquet"

april_outcome = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    AVG(gsc_impressions) AS april_avg_daily_impressions,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS april_days_observed
FROM read_parquet('{APRIL_PATH}')
WHERE gsc_data_available IS TRUE
GROUP BY client_hash_id, content_hash_id
""").df()

print("April forward-outcome table:", april_outcome.shape)

evaluated = baseline.merge(
    april_outcome, on=["client_hash_id", "content_hash_id"], how="inner"
)

dropped = len(baseline) - len(evaluated)
print(
    f"\nRows with no April data (dropped from evaluation only, "
    f"NOT from the baseline ranking itself): {dropped} "
    f"({dropped / len(baseline):.1%} of the March 31 snapshot)"
)

evaluated["future_decline_proxy"] = (
    evaluated["april_avg_daily_impressions"] < evaluated["gsc_impressions"]
).astype(int)

base_rate = evaluated["future_decline_proxy"].mean()
print(f"\nEvaluated population: {len(evaluated)} rows")
print(f"Base rate (future_decline_proxy == 1): {base_rate:.3f}")
print(
    f"A rule that scored randomly would be expected to get roughly "
    f"{base_rate:.1%} of any top-K list matching the proxy."
)


# 4. Baseline Precision@K

Same evaluated population for every K, same proxy, one table.

In [ ]:
def precision_at_k(y_true, scores, k):
    """Fraction of the top-k ranked-by-score rows where y_true == 1."""
    order = np.argsort(-np.asarray(scores), kind="stable")
    top_k = order[:k]
    return float(np.asarray(y_true)[top_k].mean())


K_VALUES = [10, 20, 50]

baseline_row = {"Method": "Week-4 baseline rule"}
for k in K_VALUES:
    baseline_row[f"Precision@{k}"] = precision_at_k(
        evaluated["future_decline_proxy"].values,
        evaluated["score"].values,
        k,
    )
baseline_row["Base Rate"] = base_rate

baseline_results = pd.DataFrame([baseline_row]).set_index("Method")
baseline_results = baseline_results[
    [f"Precision@{k}" for k in K_VALUES] + ["Base Rate"]
]

print(
    "Reading this table: compare each Precision@K to the Base Rate on the "
    "same row -- that's how much better than random selection the rule is "
    "doing at that K, against the future-performance proxy (not a "
    "ground-truth refresh label)."
)

baseline_results.round(3)


# 5. Top-10 Review

A ranked baseline should not be accepted just because it produces a neat score.

I will inspect the first ten recommendations by hand, using the actual computed
values for each row rather than a repeated template. For each item I record:

- **Action** — what the baseline recommends.
- **Why it is here** — the specific signal values that produced this rank.
- **What would make it wrong** — a plausible reason the rule could be misleading,
  chosen based on that item's own reason code.
- **Proxy outcome** — whether the item matched the future-performance proxy, shown
  for context only (the proxy was not known at ranking time).


In [ ]:
top10 = evaluated.sort_values(
    ["score", "gsc_impressions"], ascending=[False, False]
).head(10).reset_index(drop=True)

WRONG_REASON_BY_CODE = {
    "visible_and_active": (
        "high visibility and clicks may reflect a page that is already healthy "
        "and doesn't need intervention -- popularity isn't the same as opportunity"
    ),
    "visible_but_low_clicks": (
        "low clicks despite visibility could be a seasonal or SERP-feature effect "
        "(e.g. a featured snippet taking the click) rather than a content problem"
    ),
    "low_visibility": (
        "the page may serve a narrow but valuable intent where low volume is "
        "expected and reviewing it would not be a good use of limited time"
    ),
}

for _, row in top10.iterrows():
    proxy_note = (
        "matched the future-performance proxy (April avg impressions fell)"
        if row["future_decline_proxy"] == 1
        else "did NOT match the future-performance proxy (April avg impressions held or rose)"
    )
    print(
        f"Rank {int(row['rank'])} | action={row['action']} | "
        f"reason_code={row['reason_code']} | score={row['score']}\n"
        f"  Signals: gsc_impressions={row['gsc_impressions']:.0f}, "
        f"gsc_clicks={row['gsc_clicks']:.0f}, "
        f"gsc_avg_position={row['gsc_avg_position']:.1f}\n"
        f"  Why it is here: score={row['score']} from "
        f"{'meeting' if row['reason_code']!='low_visibility' else 'not meeting'} "
        f"the visibility/activity thresholds on the values above.\n"
        f"  What could make this wrong: {WRONG_REASON_BY_CODE[row['reason_code']]}.\n"
        f"  Proxy outcome (for context, not known at ranking time): {proxy_note}.\n"
    )


# 6. Weak Picks

The baseline has known weaknesses.

The biggest weakness is that high search visibility is not the same thing as
content opportunity.

The rule does not know:

- whether the content is actually outdated;
- whether search intent has changed;
- whether the page is already being worked on;
- whether traffic is seasonal;
- whether the page is strategically important;
- whether a refresh would actually improve performance.

Therefore, the baseline should be treated as a **review-prioritization heuristic**,
not an automatic refresh decision.

A future model should only be considered better if it improves the ranking using
the same evaluation design without introducing future information or leakage.


In [ ]:
from pathlib import Path

output_path = Path("work/outputs/baseline_action_score.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

queue = evaluated[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "score",
        "reason_code",
        "action",
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position",
        "report_date",
        "future_decline_proxy",
    ]
].copy()

queue.to_csv(output_path, index=False)

print(f"Queue written to: {output_path}")
print(f"Rows written: {len(queue):,}")
print("(work/**/*.csv is gitignored -- this file stays local, not committed.)")


# 7. Self-Check

- [x] Lane is Lane 2: Refresh / Content Opportunity Scoring
- [x] Existing Week-4 rule remains unchanged (visible/active thresholds, score, reason codes, actions)
- [x] Two signal checks are present (visibility, recent activity), each with a verdict
- [x] At least one signal (visibility) is linked to a real FlyRank prioritization flag
- [x] Future-performance proxy is clearly labeled as an evaluation proxy, not a refresh label
- [x] Future (April) data is used only to build the evaluation proxy, never as a baseline input
- [x] Precision@10/20/50 is calculated on the same evaluated population
- [x] Base rate is reported next to Precision@K
- [x] Top-10 review is specific per row, using each row's own computed values
- [x] No IDs or private information used as features (IDs are context/join keys only)
- [x] Notebook runs top-to-bottom (confirm in Colab: Runtime -> Run all)
- [x] Claims are careful and non-causal ("experienced the future-performance condition used by the evaluation proxy", never "needed a refresh")

## Final principle

This baseline is intentionally simple.

Its purpose is not to be impressive.

Its purpose is to establish a transparent, **numerically evaluated** benchmark that the
Week-5 model must beat honestly on the same decision setup, same proxy, and same K
values.
